In [0]:
from pyspark.sql.functions import expr
from typing import List

def process_data(df_write, tipo_carga: str, nome_gravacao_tabela: str, caminho_gravacao_tabela: str, chave_clusterby, chave_upsert: str):
    """
    Processa e grava dados em uma tabela Delta Lake com Liquid Clustering.

    Parâmetros:
    df_write (DataFrame): DataFrame para gravar.
    tipo_carga (str): 'full' ou 'delta'.
    nome_gravacao_tabela (str): Nome da tabela Delta.
    caminho_gravacao_tabela (str): Caminho onde será armazenada a tabela.
    chave_clusterby (List[str]): Lista das colunas usadas para clustering.
    chave_upsert (str): Coluna chave para operação de MERGE (upsert).

    Exemplo:
    process_data(df, 'full', 'minha_tabela', '/mnt/caminho', ['id', 'categoria'], 'id')
    """
    if tipo_carga == 'full':
        print(f"Executando carga FULL para tabela: {nome_gravacao_tabela}")
        (df_write.withColumn('data_processamento', expr("current_timestamp() - INTERVAL 3 HOURS"))
                 .write
                 .mode('overwrite')
                 .option('path', caminho_gravacao_tabela)
                 .saveAsTable(nome_gravacao_tabela))

        # Aplica o Liquid Clustering apenas na criação inicial ou alteração da tabela
        # spark.sql(f"ALTER TABLE {nome_gravacao_tabela} CLUSTER BY ({', '.join(chave_clusterby)})")

    elif tipo_carga == 'delta':
        print(f"Executando carga DELTA para tabela: {nome_gravacao_tabela}")

        if not spark.catalog.tableExists(nome_gravacao_tabela):
            print(f"Tabela {nome_gravacao_tabela} não existe no catálogo, executando carga FULL inicial.")
            
            (df_write.withColumn('data_processamento', expr("current_timestamp() - INTERVAL 3 HOURS"))
                     .write
                     .mode('overwrite')
                     .option('path', caminho_gravacao_tabela)
                     .saveAsTable(nome_gravacao_tabela))

            # Clustering inicial
            # spark.sql(f"ALTER TABLE {nome_gravacao_tabela} CLUSTER BY ({', '.join(chave_clusterby)})")

        else:
            print(f"Tabela {nome_gravacao_tabela} já existe, realizando MERGE incremental.")
            
            # Criação da view temporária para o merge
            df_write.createOrReplaceTempView("temp_view")

            colunas_source = spark.table("temp_view").columns
            colunas_update = ", ".join([f"target.{col} = source.{col}" for col in colunas_source])
            colunas_insert = ", ".join(colunas_source + ['data_processamento'])
            valores_insert = ", ".join([f"source.{col}" for col in colunas_source] + ["source.data_processamento"])

            spark.sql(f"""
                MERGE INTO {nome_gravacao_tabela} AS target
                USING (SELECT *, current_timestamp() - INTERVAL 3 HOURS AS data_processamento FROM temp_view) AS source
                ON target.{chave_upsert} = source.{chave_upsert}
                WHEN MATCHED THEN UPDATE SET {colunas_update}, target.data_processamento = source.data_processamento
                WHEN NOT MATCHED THEN INSERT ({colunas_insert}) VALUES ({valores_insert})
            """)

    else:
        raise ValueError("Parâmetro 'tipo_carga' inválido. Utilize 'full' ou 'delta'.")

    print(f"Execução finalizada: {tipo_carga.upper()}, Caminho: {caminho_gravacao_tabela}, Tabela: {nome_gravacao_tabela}")

In [0]:
def initialize_context(container_source, nome_arquivo, file_name_saida):
    """
    Initialize context for processing data.

    Parameters:
    container_source (str): The source container name.
    nome_arquivo (str): The name of the input file.
    file_name_saida (str): The name of the output file.

    Returns:
    tuple: A tuple containing:
        - var_renomear (list): List of column renaming mappings.
        - var_merge (list): List of merge conditions.
        - table_id (list): List of table IDs.
        - merge_condition (str): Merge condition string.
        - caminho_leitura (str): Path for reading data.
        - caminho_gravacao (str): Path for writing data.
        - schemalocal (str): Path for schema location.
        - checkpoint (str): Path for checkpoint.
        - nome_tabela (str): Name of the table.
    """
    var_renomear = spark.sql(f"select de, para_alias from {var_environment}.bronze.bronze_metadados_de_para_source_file where nome_arquivo = '{nome_arquivo}' ").collect()
    var_merge = spark.sql(f"select table_id, order_key from {var_environment}.bronze.bronze_metadados_source_file where nome_arquivo = '{nome_arquivo}' ").collect()
    table_id = var_merge[0]['table_id'].replace('\n', '').replace(' ', '').split(',')
    merge_condition = ' and '.join([f"s.{item.strip()} {'<>' if item.strip() == 'rastreamento_source' else '='} t.{item.strip()}" for item in var_merge[0]['table_id'].replace('\n', '').replace(' ', '').split(',')])
    caminho_leitura = f'{var_landing}/{container_source}/{nome_arquivo}/data/*/*/*'
    caminho_gravacao = f'{var_bronze}/{container_source}/{file_name_saida}/data/'
    schemalocal = f'{var_bronze}/{container_source}/{file_name_saida}/_schemalocal/'
    checkpoint = f'{var_bronze}/{container_source}/{file_name_saida}/_checkpoint/'
    nome_banco = 'bronze'
    nome_tabela = f'{var_environment}.{nome_banco}.{container_source.lower()}_{file_name_saida.lower()}'

    print(f'''
    caminho_leitura = {caminho_leitura}
    caminho_gravacao = {caminho_gravacao}
    schemalocal = {schemalocal}
    checkpoint = {checkpoint}
    nome_tabela = '{nome_tabela}'
    container_source = {container_source}
    nome_arquivo = {nome_arquivo}
    file_name_saida = {file_name_saida}
    \n
    merge = {merge_condition}
    \n
    table_id = {table_id}
    ''')

    return var_renomear, var_merge, table_id, merge_condition, caminho_leitura, caminho_gravacao, schemalocal, checkpoint, nome_tabela

In [0]:
def initialize_context_dwdistribuicao(nome_tabela):
    """
    Initialize context for processing data.

    Parameters:
    nome_tabela (str): The name of the input file.
    nome_tabela (str): The name of the output file.

    Returns:
    tuple: A tuple containing:
        - var_renomear (list): List of column renaming mappings.
        - var_merge (list): List of merge conditions.
        - table_id (list): List of table IDs.
        - merge_condition (str): Merge condition string.
        - caminho_leitura (str): Path for reading data.
        - caminho_gravacao (str): Path for writing data.
        - schemalocal (str): Path for schema location.
        - checkpoint (str): Path for checkpoint.
        - nome_tabela (str): Name of the table.
    """
    var_merge = spark.sql(f"select table_id_databricks as table_id, order_key from {var_environment}.bronze.bronze_metadados_source_to_lake where source_table_name='{nome_tabela}' ").collect()
    table_id = var_merge[0]['table_id'].replace('\n', '').replace(' ', '').split(',')
    merge_condition = ' and '.join([f"s.{item.strip()} {'<>' if item.strip() == 'rastreamento_source' else '='} t.{item.strip()}" for item in var_merge[0]['table_id'].replace('\n', '').replace(' ', '').split(',')])
    container_source = spark.sql(f"select directory_target from {var_environment}.bronze.bronze_metadados_source_to_lake where source_table_name='{nome_tabela}' ").collect()
    container_source = container_source[0]['directory_target']
    caminho_leitura = f'{var_dev}/{container_source}/{nome_tabela}/data/*/*/*'
    caminho_gravacao = f'{var_bronze}/{container_source}/{nome_tabela}/data/'
    schemalocal = f'{var_bronze}/{container_source}/{nome_tabela}/_schemalocal/'
    checkpoint = f'{var_bronze}/{container_source}/{nome_tabela}/_checkpoint/'
    nome_banco = 'bronze'
    nome_tabela = f'{var_environment}.{nome_banco}.{container_source.lower()}_{nome_tabela.lower()}'

    print(f'''
    caminho_leitura = {caminho_leitura}
    caminho_gravacao = {caminho_gravacao}
    schemalocal = {schemalocal}
    checkpoint = {checkpoint}
    nome_tabela = '{nome_tabela}'
    container_source = {container_source}
    \n
    merge = {merge_condition}
    \n
    table_id = {table_id}
    ''')

    return var_merge, table_id,container_source, merge_condition, caminho_leitura, caminho_gravacao, schemalocal, checkpoint, nome_tabela

In [0]:
def get_file_names(directory: str) -> list:
    """
    Função para obter os nomes dos arquivos em um diretório.

    Parâmetros:
    directory (str): Caminho do diretório.

    Retorna:
    list: Lista de nomes dos arquivos no diretório.

    Exemplo de uso:
    file_names = get_file_names(f"{var_landing}/sap")
    display(file_names)
    """
    files = dbutils.fs.ls(directory)
    file_names = [file.name.replace('/', '') for file in files]
    return file_names

In [0]:
def verify_table_exists(nome_tabela: str):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {nome_tabela} (
        sk_max BIGINT,
        source_table_name STRING,
        nome_tabela STRING
    )
    USING DELTA
    LOCATION '{var_bronze}/{container_source}/controller/data/'
    TBLPROPERTIES (
        delta.logRetentionDuration = 'interval 0 seconds',
        delta.autoOptimize.optimizeWrite = TRUE
    )
    """)
    return f"Tabela {nome_tabela} verificada/criada com sucesso."

In [0]:
import time

def update_dwdistribuicao_controller(table_id, nome_tabela, source_table_name, var_environment):
    ### DESABILITADO O INCREMENTAL max_sk=0 chumbado igual a 0
    # try:
    #     max_sk = spark.sql(f"""
    #     SELECT COALESCE(MAX({table_id[0]}), 0) AS max_sk
    #     FROM {nome_tabela}
    #     """).collect()[0]['max_sk']
    # except:
    #     max_sk = 0

    max_sk = 0

    # Create a DataFrame with the required data
    data = [(max_sk, source_table_name, nome_tabela)]
    df = spark.createDataFrame(data, ['max_sk', 'source_table_name', 'nome_tabela'])

    df.createOrReplaceTempView('updates')

    for _ in range(10):  # Retry up to 10 times
        try:
            spark.sql(f"""
            MERGE INTO {var_environment}.bronze.dwdistribuicao_controller AS target
            USING updates AS source
            ON target.source_table_name = source.source_table_name
            WHEN MATCHED THEN
              UPDATE SET target.sk_max = source.max_sk
            WHEN NOT MATCHED THEN
              INSERT (sk_max, source_table_name, nome_tabela) 
              VALUES (source.max_sk, source.source_table_name, source.nome_tabela)
            """)
            print(f"Operação realizada: table_id, valor: {max_sk}")
            break
        except Exception as e:
            if "ConcurrentAppendException" in str(e):
                print("ConcurrentAppendException detected, retrying...")
                time.sleep(2)
            else:
                raise e

In [0]:
def execute_bronze_table_update(var_environment: str):
    df_bronze = spark.read.table(f"{var_environment}.bronze.dwdistribuicao_controller")
    df_bronze.coalesce(1).write.mode('overwrite').format('delta').option('delta.logRetentionDuration', 'interval 0 hours').saveAsTable(f"{var_environment}.bronze.dwdistribuicao_controller")
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled = false")
    spark.sql(f"VACUUM {var_environment}.bronze.dwdistribuicao_controller RETAIN 0 HOURS")

In [0]:
def upsertToDeltaLive_order_key(nome_tabela: str, caminho_gravacao: str, merge_condition: str, table_id: list, order_key: str ):
    """
    Função para realizar upsert (inserção ou atualização) em uma tabela Delta Live.

    Parâmetros:
    nome_tabela (str): Nome da tabela Delta.
    caminho_gravacao (str): Caminho onde a tabela Delta será gravada.
    merge_condition (str): Condição de merge para o upsert.
    table_id (list): Lista de colunas que identificam unicamente os registros na tabela.
    order_key (str): Coluna utilizada para ordenar os registros.

    Retorna:
    function: Função que realiza o upsert em micro-batches.

    Exemplo de uso:
    nome_tabela = "minha_tabela_delta"
    caminho_gravacao = "/mnt/data/delta/minha_tabela"
    merge_condition = "t.id = s.id"
    table_id = ["id"]
    order_key = "timestamp"
    upsert_func = upsertToDeltaLive(nome_tabela, caminho_gravacao, merge_condition, table_id, order_key)
    df = spark.read.parquet("/mnt/data/landing/sample.parquet")
    upsert_func(df, 1)
    """
    def _do_work(microbatchOutputDF, batchId):
        if(spark.catalog.tableExists(f'{nome_tabela}')):
            print('Estou executando insert or update')
            microbatchOutputDF = add_ingestion_date(microbatchOutputDF)
            microbatchOutputDF = microbatchOutputDF.withColumn("rn2", row_number()\
                .over(Window.partitionBy([col(x) for x in table_id])\
                .orderBy(microbatchOutputDF[order_key].desc())))
            microbatchOutputDF = microbatchOutputDF.filter('rn2 = 1')
            microbatchOutputDF = microbatchOutputDF.drop(col('rn2'))
            deltadf = DeltaTable.forName(spark, f'{nome_tabela}')
            (deltadf.alias('t')
             .merge(
                 microbatchOutputDF.alias('s'),
                 f"{merge_condition}")
             .whenMatchedUpdateAll(condition=f"s.order_key >= t.order_key")
             .whenNotMatchedInsertAll()
             .execute()
            )
        else:
            print('Estou executando do inicio')
            microbatchOutputDF = add_ingestion_date(microbatchOutputDF)
            microbatchOutputDF = microbatchOutputDF.withColumn("rn2", row_number()\
                .over(Window.partitionBy([col(x) for x in table_id])\
                .orderBy(microbatchOutputDF[order_key].desc())))
            microbatchOutputDF = microbatchOutputDF.filter('rn2 = 1')
            microbatchOutputDF = microbatchOutputDF.drop(col('rn2'))
            print('Primeira Ingestão')
            microbatchOutputDF.write.mode('overwrite').option('path', f'{caminho_gravacao}')\
                               .option("Mergeschema", "True")\
                               .format('delta').saveAsTable(f'{nome_tabela}')
    return _do_work

In [0]:
def upsertToDeltaLive(nome_tabela: str, caminho_gravacao: str, merge_condition: str, table_id: list, order_key: str):
    """
    Função para realizar upsert (inserção ou atualização) em uma tabela Delta Live.

    Parâmetros:
    nome_tabela (str): Nome da tabela Delta.
    caminho_gravacao (str): Caminho onde a tabela Delta será gravada.
    merge_condition (str): Condição de merge para o upsert.
    table_id (list): Lista de colunas que identificam unicamente os registros na tabela.
    order_key (str): Coluna utilizada para ordenar os registros.

    Retorna:
    function: Função que realiza o upsert em micro-batches.

    Exemplo de uso:
    nome_tabela = "minha_tabela_delta"
    caminho_gravacao = "/mnt/data/delta/minha_tabela"
    merge_condition = "t.id = s.id"
    table_id = ["id"]
    order_key = "timestamp"
    upsert_func = upsertToDeltaLive(nome_tabela, caminho_gravacao, merge_condition, table_id, order_key)
    df = spark.read.parquet("/mnt/data/landing/sample.parquet")
    upsert_func(df, 1)
    """
    def _do_work(microbatchOutputDF, batchId):
        if(spark.catalog.tableExists(f'{nome_tabela}')):
            print('Estou executando insert or update')
            microbatchOutputDF = add_ingestion_date(microbatchOutputDF)
            microbatchOutputDF = microbatchOutputDF.withColumn("rn2", row_number()\
                .over(Window.partitionBy([col(x) for x in table_id])\
                .orderBy(microbatchOutputDF[order_key].desc())))
            microbatchOutputDF = microbatchOutputDF.filter('rn2 = 1')
            microbatchOutputDF = microbatchOutputDF.drop(col('rn2'))
            deltadf = DeltaTable.forName(spark, f'{nome_tabela}')
            (deltadf.alias('t')
             .merge(
                 microbatchOutputDF.alias('s'),
                 merge_condition)
             .whenMatchedUpdateAll()
             .whenNotMatchedInsertAll()
             .execute()
            )
        else:
            print('Estou executando do inicio')
            microbatchOutputDF = add_ingestion_date(microbatchOutputDF)
            microbatchOutputDF = microbatchOutputDF.withColumn("rn2", row_number()\
                .over(Window.partitionBy([col(x) for x in table_id])\
                .orderBy(microbatchOutputDF[order_key].desc())))
            microbatchOutputDF = microbatchOutputDF.filter('rn2 = 1')
            microbatchOutputDF = microbatchOutputDF.drop(col('rn2'))
            print('Primeira Ingestão')
            microbatchOutputDF.write.mode('overwrite').option('path', f'{caminho_gravacao}')\
                               .option("mergeSchema", "true")\
                               .format('delta').saveAsTable(f'{nome_tabela}')
    return _do_work

In [0]:
def get_parquet_files_and_create_df(directory_path: str, tipo: str, camada_leitura: str, camada_destino: str) -> DataFrame:
    """
    Função para obter arquivos Parquet de um diretório e criar um DataFrame com informações de leitura, nome do arquivo, destino, camada de leitura e camada de destino.

    Parâmetros:
    directory_path (str): Caminho do diretório onde os arquivos Parquet estão localizados.
    tipo (str): Tipo de arquivo a ser procurado (por exemplo, 'parquet').
    camada_leitura (str): Nome da camada de leitura (por exemplo, 'landing').
    camada_destino (str): Nome da camada de destino (por exemplo, 'bronze').

    Retorna:
    DataFrame: DataFrame contendo as colunas 'leitura', 'arquivo', 'destino', 'camada_leitura' e 'camada_destino'.

    Exemplo de uso:
    directory_path = "/mnt/data/landing"
    tipo = "parquet"
    camada_leitura = "landing"
    camada_destino = "bronze"
    df = get_parquet_files_and_create_df(directory_path, tipo, camada_leitura, camada_destino)
    display(df)
    """
    arquivo_files = []
    files_to_treat = dbutils.fs.ls(directory_path)
    while files_to_treat:
        path = files_to_treat.pop(0).path
        if path.endswith('/'):
            files_to_treat += dbutils.fs.ls(path)
        elif path.endswith(f'.{tipo}'):
            arquivo_files.append(path)
    df = spark.createDataFrame([(item, item.split('/')[-1].replace(f'.{tipo}', ''), item.replace(camada_leitura, camada_destino).replace(f'.{tipo}', ''), camada_leitura, camada_destino) for item in arquivo_files], ["leitura", "arquivo", "destino", "camada_leitura", "camada_destino"])
    return df

In [0]:
def gravar_parquet_unico(environment: str, leitura: str, arquivo: str, destino: str, camada_leitura: str, camada_destino: str) -> None:
    """
    Função para ler um arquivo Parquet, adicionar uma coluna de data de processamento e gravar o DataFrame resultante em uma tabela.

    Esta função é utilizada junto com a função get_parquet_files_and_create_df.

    Parâmetros:
    environment (str): Nome do ambiente (por exemplo, 'dev', 'prod').
    leitura (str): Caminho do arquivo Parquet a ser lido.
    arquivo (str): Nome do arquivo sem extensão.
    destino (str): Caminho de destino onde o DataFrame será salvo.
    camada_leitura (str): Nome da camada de leitura (por exemplo, 'landing').
    camada_destino (str): Nome da camada de destino (por exemplo, 'bronze').

    Retorna:
    None

    Exemplo de uso:
    environment = "dev"
    leitura = "/mnt/data/landing/sample.parquet"
    arquivo = "sample"
    destino = "/mnt/data/bronze/sample"
    camada_leitura = "landing"
    camada_destino = "bronze"
    gravar_parquet_unico(environment, leitura, arquivo, destino, camada_leitura, camada_destino)
    """
    df = spark.read.parquet(leitura)
    df = df.withColumn("data_processamento", expr("current_timestamp() - INTERVAL 3 HOURS"))
    df.write.mode("overwrite").saveAsTable(f"{environment}.{camada_destino}.{camada_destino}_{arquivo}", path=destino)

In [0]:
def adicionaComentariosTabela(catalogoUnity: str, nomeSchema: str, nomeTabela: str, comentarioTabela: str, comentariosColunas: dict) -> None:
    """
    Adiciona comentários a uma tabela e suas colunas no PySpark.
    
    Parâmetros:
        catalogoUnity (str): O catálogo onde a tabela está localizada.
        nomeSchema (str): O esquema onde a tabela está localizada.
        nomeTabela (str): O nome da tabela.
        comentarioTabela (str): O comentário a ser adicionado à tabela.
        comentariosColunas (dict): Um dicionário onde as chaves são nomes das colunas e os valores são os comentários para essas colunas.
    
    Exemplo de código utilizando a função:
        catalogoUnity = "dev"
        nomeSchema = "bronze"
        nomeTabela = "nome_test"
        comentarioTabela = "Tabela de teste com comentários"
        comentariosColunas = {
            "id": "Identificador único",
            "nome": "Nome da pessoa"
        }
        adicionaComentariosTabela(catalogoUnity, nomeSchema, nomeTabela, comentarioTabela, comentariosColunas)
    """
    spark.sql(f"COMMENT ON TABLE {catalogoUnity}.{nomeSchema}.{nomeTabela} IS '{comentarioTabela}'")
    for coluna, comentario in comentariosColunas.items():
        sqlquery = f"ALTER TABLE {catalogoUnity}.{nomeSchema}.{nomeTabela} CHANGE COLUMN {coluna} COMMENT '{comentario}'"
        spark.sql(sqlquery)
        print(f"Comentário: <{comentario}> da coluna <{coluna}> alterado com sucesso!")

In [0]:
from pyspark.sql import DataFrame

def add_ingestion_date(input_df: DataFrame) -> DataFrame:
    """
    Função para adicionar uma coluna de data de processamento a um DataFrame.

    Parâmetros:
    input_df (DataFrame): DataFrame de entrada.

    Retorna:
    DataFrame: DataFrame com a coluna 'data_processamento' adicionada.

    Exemplo de uso:
    df = spark.read.parquet("/mnt/data/landing/sample.parquet")
    df_com_data = add_ingestion_date(df)
    display(df_com_data)
    """
    output_df = input_df.withColumn("data_processamento", expr("current_timestamp() - INTERVAL 3 HOURS"))
    return output_df

In [0]:
from pyspark.sql.utils import AnalysisException

def optimize_tables(catalogo_unity: str, nome_schema: str, nome_tabela: str) -> None:
    """
    Executa a manutenção de uma tabela Delta Lake no Unity Catalog,
    incluindo: OPTIMIZE, REFRESH TABLE e VACUUM com retenção de 7 dias.

    Parâmetros:
        catalogo_unity (str): Nome do catálogo (ex: 'dev').
        nome_schema (str): Nome do schema (ex: 'bronze').
        nome_tabela (str): Nome da tabela Delta (ex: 'minha_tabela').

    Exemplo:
        otimizacao_tabela("dev", "bronze", "minha_tabela")
    """
    full_table_name = f"{catalogo_unity}.{nome_schema}.{nome_tabela}"

    try:
        print(f"Iniciando manutenção da tabela: {full_table_name}")

        # Verifica se a tabela existe
        if not spark.catalog.tableExists(full_table_name):
            print(f"Erro: Tabela {full_table_name} não encontrada.")
            return

        # Ativa autoMerge
        spark.sql("SET spark.databricks.delta.schema.autoMerge.enabled = true")
        # print("Configuração 'autoMerge' ativada.")

        # Executa OPTIMIZE
        # print("Executando OPTIMIZE...")
        spark.sql(f"OPTIMIZE {full_table_name}")
        # print("OPTIMIZE concluído com sucesso.")

        # Executa VACUUM com retenção de 7 dias
        # print("Executando VACUUM com retenção de 168 horas (7 dias)...")
        spark.sql(f"VACUUM {full_table_name} RETAIN 168 HOURS")
        # print("VACUUM concluído com sucesso.")

        # Executa REFRESH TABLE
        # print("Executando REFRESH TABLE...")
        spark.sql(f"REFRESH TABLE {full_table_name}")
        # print("Metadados atualizados com sucesso.")

        # Resumo da operação
        print(f"Resumo da otimização para a tabela {full_table_name}: Configuração 'autoMerge' ativada, OPTIMIZE executado com sucesso, Metadados atualizados com REFRESH TABLE, VACUUM executado com retenção de 7 dias.")

    except AnalysisException as ae:
        print(f"Erro de análise: {ae}")
    except Exception as e:
        print(f"Erro inesperado durante a manutenção da tabela: {e}")

In [0]:
def setShufflePartitions(qtd_shuffle: int, ignoreMissingFiles: bool) -> str:
    """
    Configura as partições de shuffle e a opção de ignorar arquivos ausentes no Spark.

    Parâmetros:
        qtd_shuffle (int): A quantidade de partições de shuffle a ser configurada.
        ignoreMissingFiles (bool): Se deve ignorar arquivos ausentes.

    Retorna:
        str: Uma string confirmando as configurações aplicadas.
    """
    spark.conf.set("spark.sql.shuffle.partitions", str(qtd_shuffle))
    spark.conf.set("spark.sql.files.ignoreMissingFiles", str(ignoreMissingFiles).lower())
    return f'shuffle:{qtd_shuffle}, ignore missing files:{ignoreMissingFiles}'

In [0]:
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Função: SCD2_MERGE()
# Autor: Cristiano Moraes 
# Data: 19/02/2025
# Descrição: Realiza a atualização de uma tabela delta de dimensão onde TODOS os campos são versionados - SCD Type 2
#
# Parâmetros:
# source_df: DataFrame com os dados a serem atualizados (e.g Dataframe que contém os dados incrementais, sempre pegar o registro ativo pelo 'rastreamento_source')
#            Quando for preparar este dataframe, usar RANK() OVER(PARTITION BY a.cdCliente ORDER BY a.rastreamento_source DESC) as rank  e filtrar rank=1   
# target_dim_name: Nome da tabela delta onde será realizada a atualização (e.g dim_cliente)
# natural_key: Chave natural da dimensão (e.g CodCliente)
# surrogate_key: Coluna surrogate key (e.g SkCliente)
# flag_active_column: Coluna que indica se o registro é ativo ou não (e.g IndRegistroAtual)
# updated_timestamp_column: Coluna que indica o timeStamp de atualização, usado no mecanismo do Merge SCD 2 (e.g DataAtualizacao)
# compare_columns: Lista de colunas que devem ser comparadas para verificar se houve alteração para versionamento (e.g ['NomeCliente', 'CodRede', 'CodAssociacao'])
# ignore_column_list: Lista de colunas que devem ser ignoradas na comparação (e.g ['GerenteAtual']) // tudo que tiver aqui não vai ser "versionado" (pode passar lista [] vazia)
#
# exemplo: 
#       campos_versionar = ["NomeCliente"]
#       scd2_merge(df_incremental, caminho_gravacao_tabela, "CodCliente", "SkCliente", "InRegistroAtivo", "DataAtualizacao", campos_versionar, [])          
#
# ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.functions import lit, max as spark_max, row_number
from pyspark.sql.types import StringType

def scd2_merge(source_df, target_dim_name, natural_key, surrogate_key, flag_active_column, updated_timestamp_column,compare_columns=[], ignore_column_list=[]):
    #1 - Le a dimensão atual do delta, para comparar com o source_df  
    target_table = DeltaTable.forPath(spark, target_dim_name) 
      
    #2 - Gera o SK MAX()+1, no caso de alguma alteração nos dados (Aqui eu garanto a geração de novas surrogate kay (SK), para registros novos)
    max_sk = target_table.toDF().agg(spark_max(surrogate_key)).collect()[0][0]
    if max_sk is None:
        max_sk = 0  # Default é 0 se sk não existe
    window_spec = Window.orderBy(natural_key)  # Ordenamento para garantir a consistencia da SK
    source_sk = source_df.withColumn(
    surrogate_key, 
    (lit(max_sk) + row_number().over(window_spec)).cast("string")  # cast para string
    )

    #3 Código genérico que compara os valores de todas as colunas (Excluindo SK, natural key, colunas de controle e lista customizada passada por parâmetro)
    col_data_processamento = "data_processamento"
    ignore_default_columns = [surrogate_key, flag_active_column, natural_key, updated_timestamp_column, col_data_processamento] 
    all_columns = [c for c in source_sk.columns if c not in ignore_default_columns]
    ignore_columns = ignore_default_columns + ignore_column_list
    if len(compare_columns) == 0:
        compare_columns = [c for c in source_sk.columns if c not in ignore_columns]

        # monta a clausula de comparação para ver se alguma coluna versionável mudou comparando a dimensão x source_df
    change_condition = " OR ".join([f"target.{c} <> source.{c}" for c in compare_columns])

    #4 MERGE SCD Type 2
    join_condition = (
        (col(f"target.{flag_active_column}") == 1) & 
        (col(f"target.{natural_key}") == col(f"source.{natural_key}")) &     
        reduce(lambda a, b: a | b, [col(f'target.{c}') != col(f'source.{c}') for c in compare_columns])
    )

    insert_values = {
        surrogate_key: f"source.{surrogate_key}",        
        natural_key: f"source.{natural_key}",       
        **{c: f"source.{c}" for c in all_columns},  
        flag_active_column: lit(1),  # sempre 1 porque este vai passar a ser o registro ativo 
        updated_timestamp_column: from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"),
        col_data_processamento: from_utc_timestamp(current_timestamp(), "America/Sao_Paulo")
    }

        # Se o registro existe (chave natural) e está ativo, atualizo o status/flag para inativo=0. Se não existe, insere registro novo na dimensão
    target_table.alias("target").merge(
        source_sk.alias("source"),
        f"target.{natural_key} = source.{natural_key} AND target.{flag_active_column} = 1"                    
    ).whenMatchedUpdate(    ## invalida o antigo ativo colocando 0
        condition=change_condition,
        set={
            flag_active_column: lit(0),  
            updated_timestamp_column: from_utc_timestamp(current_timestamp(), "America/Sao_Paulo")
        }        
    ).whenNotMatchedInsert(  ## REGISTRO NOVO (Não tem chave natural) OU tem chave natural e flag_ativo=0 (vai inserir vou está "ativando" ele)
        values=insert_values
    ).execute()

        # Aqui vai inserir a nova versão (flag_Ativo=1) dos que setou flag_ativo=0 no passo anterior
    target_table.alias("target").merge(
        source_sk.alias("source"),
        f"target.{natural_key} = source.{natural_key} AND target.{flag_active_column} = 1"                    
    ).whenNotMatchedInsert(  
        values=insert_values
    ).execute()

In [0]:
def process_fact(df_write, nome_gravacao_tabela, caminho_gravacao_tabela, data_formatada, chave_clusterby):

    df_write = df_write.withColumn('data_processamento', from_utc_timestamp(current_timestamp(), "America/Sao_Paulo")) 
    df_write = df_write.withColumn("data_formatada", F.to_date(F.col(data_formatada).cast("string"), "yyyyMMdd")) 

    df_write.write.format("delta").mode("overwrite").option('path', caminho_gravacao_tabela).saveAsTable(nome_gravacao_tabela)
    print(f"Carga histórica da fato {nome_gravacao_tabela} completa. {df_write.count()} registros inseridos.")

    spark.sql(f"ALTER TABLE {nome_gravacao_tabela} CLUSTER BY ({', '.join(chave_clusterby)})")
    spark.sql(f"OPTIMIZE {nome_gravacao_tabela}")
    spark.sql(f"VACUUM {nome_gravacao_tabela}")

In [0]:
def drop_table(tabela):
    try:
        spark.sql(f"DROP VIEW IF EXISTS {tabela}")
    except:
        location = spark.sql(f"DESCRIBE DETAIL {tabela}").collect()[0]['location']
        spark.sql(f"DROP TABLE IF EXISTS {tabela}")
        dbutils.fs.rm(location, recurse=True)
        try:
            dbutils.fs.rm(location.replace('/data', '/_schemalocal'), recurse=True)
            dbutils.fs.rm(location.replace('/data', '/_checkpoint'), recurse=True)
            # Remove o diretório pai da tabela
            parent_dir = '/'.join(location.rstrip('/').split('/')[:-1])
            dbutils.fs.rm(parent_dir, recurse=True)
        except:
            pass


In [0]:
def drop_feature_v2checkpoint(nome_gravacao_tabela):
    try:
        spark.sql(f'''
        ALTER TABLE {nome_gravacao_tabela} DROP FEATURE v2Checkpoint TRUNCATE HISTORY; 
        ''')
    except Exception as e:
        print(f"Erro ao executar ALTER TABLE: {nome_gravacao_tabela} DROP FEATURE v2Checkpoint TRUNCATE HISTORY: {e}")

In [0]:
"""
----------------------------------------------------------------------------------------------------------------
 Função: incrementalToDeltaLiveByDate()
 Autor: Jose Vilson da Cruz 
 Data: 26/06/2025
 Ingestão incremental para camada bronze substituindo dados históricos a partir da menor data do batch novo.

    Parâmetros:
    - nome_tabela (str): Nome da tabela Delta.
    - caminho_gravacao (str): Caminho de gravação da tabela Delta.
    - date_id (str): Nome da coluna de data (ex: 'data_venda').
-------------------------------------------------------------------------------------------------------------------   
"""
from pyspark.sql.functions import col, min as spark_min, max as spark_max
from delta.tables import DeltaTable
def incrementalToDeltaLiveByDate(nome_tabela: str, caminho_gravacao: str, date_id: str):
    """
    Função de alto nível para processamento incremental com foreachBatch, usando MERGE INTO.
    Esta função retorna outra função, que é executada pelo Spark a cada microbatch.
    """
    def _do_work(microbatchOutputDF, batchId):
        """
        Esta função interna é executada a cada microbatch do stream.
        Ela processa os arquivos de forma ordenada e usa MERGE INTO
        para atualizar ou inserir dados no Delta Table.
        """
        # Ignora microbatches vazios para evitar operações desnecessárias.
        if microbatchOutputDF.head(1) is None:
            print(f"[Lote {batchId}] Microbatch vazio. Nada a fazer.")
            return

        # Coleta os nomes de arquivos únicos e os ordena para garantir
        # que sejam processados na sequência desejada (por exemplo, cronológica).
        arquivos = sorted([r[0] for r in microbatchOutputDF.select("rastreamento_source").distinct().collect()])
        
        # Verifica se a tabela Delta de destino já existe.
        tabela_existe = spark.catalog.tableExists(nome_tabela)

        if not tabela_existe:
            # Se a tabela não existe, processa cada arquivo individualmente.
            # O primeiro arquivo cria a tabela (modo 'overwrite'), e os seguintes
            # são mesclados para evitar duplicatas.
            print(f"[Lote {batchId}] Tabela Delta '{nome_tabela}' não existe. Criando nova tabela...")
            
            for i, arquivo in enumerate(arquivos):
                df_arquivo = microbatchOutputDF.filter(col("rastreamento_source") == arquivo)
                
                if i == 0:
                    # Primeiro arquivo: cria a tabela em modo 'overwrite'
                    df_arquivo.write.format("delta") \
                        .mode("overwrite") \
                        .option("path", caminho_gravacao) \
                        .saveAsTable(nome_tabela)
                    print(f"[Lote {batchId}] Tabela '{nome_tabela}' criada com o arquivo inicial: '{arquivo}'")
                else:
                    # Arquivos seguintes: mescla os dados usando MERGE INTO
                    # Calcula a menor date_id do microbatch
                    menor_data = df_arquivo.select(spark_min(col(date_id))).first()[0]
                    print(f"[Lote {batchId}] Menor {date_id} no microbatch: {menor_data}")
                    # Calcula a maior date_id do microbatch
                    maior_data = df_arquivo.select(spark_max(col(date_id))).first()[0]
                    print(f"[Lote {batchId}] Maior {date_id} no microbatch: {maior_data}")

                    # Acessa a tabela Delta e remove dados históricos com data >= menor_data
                    delta_table = DeltaTable.forName(spark, nome_tabela)
                    query = f"{date_id} >= '{menor_data}' AND {date_id} <= '{maior_data}'"
                    delta_table.delete(query)

                    df_arquivo.write.format("delta") \
                        .mode("append") \
                        .option("path", caminho_gravacao) \
                        .save()
                    print(f"[Lote {batchId}] Dados do arquivo '{arquivo}' adicionados com sucesso.")
        else:
            # Se a tabela já existe, processa cada arquivo ordenado
            # usando a lógica de DELETE e APPEND.
            print(f"[Lote {batchId}] Tabela '{nome_tabela}' já existe. Iniciando o processamento de {len(arquivos)} arquivo(s)...")
            
            for arquivo in arquivos:
                # Filtra o DataFrame do microbatch para processar apenas um arquivo por vez
                df_arquivo = microbatchOutputDF.filter(col("rastreamento_source") == arquivo)
                
                # Calcula a menor date_id do arquivo
                menor_data = df_arquivo.select(spark_min(col(date_id))).first()[0]
                print(f"[Lote {batchId}] Menor {date_id} no arquivo: {menor_data}")
                
                # Calcula a maior date_id do arquivo
                maior_data = df_arquivo.select(spark_max(col(date_id))).first()[0]
                print(f"[Lote {batchId}] Maior {date_id} no arquivo: {maior_data}")

                # Acessa a tabela Delta e remove o intervalo de datas do arquivo
                delta_table = DeltaTable.forName(spark, nome_tabela)
                query = f"{date_id} >= '{menor_data}' AND {date_id} <= '{maior_data}'"
                delta_table.delete(query)

                # Insere os dados do arquivo na tabela
                df_arquivo.write.format("delta") \
                    .mode("append") \
                    .option("path", caminho_gravacao) \
                    .save()
                print(f"[Lote {batchId}] Arquivo '{arquivo}' processado com sucesso.")

    return _do_work

In [0]:
def merge_tabela_delta(
    nome_catalogo: str,
    nome_gravacao_tabela: str,
    surrogate_key: str,
    merge_field: str
):
  """
    Realiza uma operação de MERGE (upsert + delete) entre uma tabela Delta existente
    e uma nova temporary view no Databricks, utilizando SQL dinâmico.
    A função:
    - Atualiza registros existentes com base no campo de junção (`merge_field`)
    - Insere novos registros com um surrogate key incremental e campo `data_processamento` (com timestamp atual ajustado para UTC-3)
    - Remove registros da tabela de destino que não estão na origem (se a surrogate key for maior que zero)
    Requisitos:
    - A temporary view `new_<nome_gravacao_tabela>` deve existir no contexto do Spark antes da execução.
    - A tabela destino (`<nome_catalogo>.<nome_gravacao_tabela>`) deve conter as mesmas colunas da origem, além da coluna `data_processamento`.
    Parâmetros:
        nome_catalogo (str): Nome completo do catálogo/schema onde está a tabela destino (ex: 'meu_catalogo.meu_schema').
        nome_gravacao_tabela (str): Nome base da tabela que será atualizada/mesclada (sem prefixo).
        surrogate_key (str): Nome da chave substituta (ID incremental) da tabela.
        merge_field (str): Campo de junção (chave de negócio) para comparar os registros da origem e destino.
    Retorno:
        DataFrame com o resultado da execução do comando MERGE via spark.sql().
  """
  # Obtém colunas da nova tabela, exceto a surrogate_key
  cols = spark.table(f"new_{nome_gravacao_tabela}").columns
  cols_sem_surrogate = [col for col in cols if col != surrogate_key]
  cols_sql = ", ".join([f"new.{c}" for c in cols_sem_surrogate])
  query = f'''
  MERGE INTO
    {nome_catalogo}.{nome_gravacao_tabela} AS target
  USING (
    --UPDATE Records
    SELECT
      old.{surrogate_key},
      {cols_sql}, old.data_processamento
    FROM
      {nome_catalogo}.{nome_gravacao_tabela} old
    INNER JOIN
      new_{nome_gravacao_tabela} new
      ON old.{merge_field} = new.{merge_field}

    UNION ALL
    
    --INSERT Records
    SELECT
      row_number() OVER(ORDER BY {merge_field}) +
      (SELECT COALESCE(MAX({surrogate_key}), 0) FROM {nome_catalogo}.{nome_gravacao_tabela}) AS {surrogate_key},
      {cols_sql}, (current_timestamp() - INTERVAL 3 HOURS) as data_processamento
    FROM
      new_{nome_gravacao_tabela} new
    WHERE
      {merge_field} NOT IN (SELECT {merge_field} FROM {nome_catalogo}.{nome_gravacao_tabela})
  ) AS source
  ON
    target.{merge_field} = source.{merge_field}
  WHEN MATCHED THEN
    UPDATE SET *
  WHEN NOT MATCHED THEN 
    INSERT *
  WHEN NOT MATCHED BY SOURCE AND {surrogate_key} > 0 THEN
    DELETE
  '''
  return spark.sql(query)

In [0]:
class JsonPerEventHandler(logging.Handler):
    """
    Cria um arquivo json por evento registrado no logger no diretório fornecido (base_dir_adls).
    Layout -> base_dir_adls/data/YYYY/MM/DD/<notebook>-HH:MM:SS.json
    Sendo: YYYY, MM, DD, HH, MM, SS, ddd -> Campos extraídos da data/hora atual usando o timezone fornecido: datetime_timezone().

    1) Parâmetros:
        base_dir_adls (str): diretório onde os arquivos serão criados.
        
    2) Parâmetro do job:
        "job_id" -> {{job.id}}
        "job_name" -> {{job.name}}
        "job_run_id" -> {{job.run_id}}

    3) Exemplo de uso: (define base_dir_adls e limpa os handlers antes de incluir o handler criado)

        BASE_DIR_ADLS = "abfss://meucontainer@contaadlsexemplo.dfs.core.windows.net/monitoring/logs"

        logger = logging.getLogger("monitor")
        logger.setLevel(logging.INFO)
        for h in list(logger.handlers):
            logger.removeHandler(h)

        logger.addHandler(JsonPerEventHandler(BASE_DIR_ADLS))
        log_level = "INFO"

        log_data = {
        "evt": "inicio_run",
        "notebook": "silver_dimensao_canal_vendas",
        "elapsed_time": 50,
        "error": None,
        }

        level = getattr(logging, log_level)
        record = logging.LogRecord(
            name="monitor", 
            level=level,  # Nível dinâmico de log
            pathname="", 
            lineno=0, 
            msg="", 
            args=(), 
            exc_info=None
        )
        record.payload = log_data
        logger.handle(record)

    4) Informações contidas no json gerado:

        a. "levelname"
        b. "data_execucao"
        c. "evt" => se incluído no payload.
        d. "notebook" => se incluído no payload, caso contrário "unknown".
        e. "job_id" => se tiver o parâmetro de job.
        f. "job_name" => se tiver o parâmetro de job.
        g. "job_run_id" => se tiver o parâmetro de job.
        h. "elapsed_time" => se incluído no payload.
        i. "error" => se incluído no payload.
    """
    def __init__(self, base_dir_adls: str):
        super().__init__()
        self.base_dir = base_dir_adls
        self.job_id = None
        self.job_name = None
        self.job_run_id = None

    def datetime_timezone(self, tz: str = "America/Sao_Paulo"):
        """
        Retorna datetime.now() com timezone. Se não informado, usa 'America/Sao_Paulo'
        
        Parâmetros:
        tz (str): Localização para obter timezone.

        Retorna:
        Datetime aplicando o timezone informado.

        Exemplo de uso:
        
        data_agora = datetime_timezone(tz = "America/Los_Angeles")
        print(data_agora)    
        """
        try:
            from zoneinfo import ZoneInfo
            tz_ = ZoneInfo(tz)
        except Exception:
            import pytz
            tz_ = pytz.timezone(tz)
        return datetime.now(tz_)

    def _safe_basename(self, name: str) -> str:
        """
        Pega o último componente de caminho de um arquivo, removendo diretórios. Convertendo para texto se não for. 
        Substitui caracteres diferentes de números, letras, hífen, ponto e underline pelo caractere underline.
        """
        base = os.path.basename(str(name)).strip()
        return re.sub(r"[^A-Za-z0-9_.-]+", "_", base)

    def _datetime_converter(self, obj):
        """
        Converte um datetime para ISO 8601.
        """
        if isinstance(obj, datetime):
            return obj.isoformat()
        raise TypeError("Type not serializable")

    def _get_task_info(self):
        """
        Captura somente o que vier por widgets do Job. Prioriza os parâmetros do Job atual (sufixo '2').
        Se não existirem, busca o parâmetro sem o sufixo e se não existir, retorna None.
        """
        def _get_parametro(chave2,chave1):
            try:
                return dbutils.widgets.get(chave2)
            except Exception:
                try:
                    return dbutils.widgets.get(chave1)
                except Exception:
                    return None
    
        self.job_id = _get_parametro("job_id2","job_id")
        self.job_name = _get_parametro("job_name2","job_name")
        self.job_run_id = _get_parametro("job_run_id2","job_run_id")

        return self.job_id, self.job_name, self.job_run_id

    def _ensure_dir(self, dir_path: str):
        """
        Cria um diretório se não existir.
        """
        dbutils.fs.mkdirs(dir_path)

    def _time_for_filename(self, dt: datetime) -> str:
        """
        Retorna hora como string no formato "HH:MM:SS.ddd" a partir de um datetime fornecido.
        """
        return dt.strftime("%H:%M:%S.") + f"{int(dt.microsecond/1000):03d}"

    def _dir_for(self, dt: datetime) -> str:
        """
        Retorna o caminho completo do diretório para o log.
        A estrutura é "data/YYYY/MM/DD"
        """
        subdir = os.path.join("data", dt.strftime("%Y/%m/%d"))
        return os.path.join(self.base_dir, subdir)

    def emit(self, record: logging.LogRecord):
        """
        Processa o evento de log e grava um arquivo JSON separado para cada execução, mantendo apenas campos pertinentes.
        """
        try:
            job_id, job_name, job_run_id = self._get_task_info()

            payload = getattr(record, "payload", {}) or {}

            notebook_name = payload.get("notebook") or "unknown"
            notebook_name = self._safe_basename(notebook_name)

            dt_now = self.datetime_timezone()
            dir_path = self._dir_for(dt_now)
            self._ensure_dir(dir_path)

            fname = f"{notebook_name}-{self._time_for_filename(dt_now)}.json"
            fpath = os.path.join(dir_path, fname)

            log_data = {
                "levelname": record.levelname,
                "data_execucao": dt_now.isoformat(),
                "evt": payload.get("evt"),
                "notebook": notebook_name,
                "job_id": payload.get("job_id", self.job_id),
                "job_name": payload.get("job_name", self.job_name),
                "job_run_id": payload.get("job_run_id", self.job_run_id),
                "elapsed_time": payload.get("elapsed_time" ),
                "error": payload.get("error")
            }

            body = json.dumps(log_data, default=self._datetime_converter, ensure_ascii=False)
            
            dbutils.fs.put(fpath, body, overwrite=False)

        except Exception as e:
            print(f"Falha ao gravar log por evento: {e}")

def log_event(level: str, **payload):
    """
    Envia o registro para o logger, com o level e o payload fornecidos.
    logRecord é preenchido com "nulo" em tudo, só sendo utilizado para incluir o log com o level.

    Parâmetros
    level (str): "INFO", "WARNING", "ERROR", etc.
    payload (dict): Dicionário com os campos do log.

    Exemplo de uso: (esse formato não é muito bom para uso direto, pois não está fazendo o uso apropriado do logRecord. Recomendado o uso junto com a classe criada JsonPerEventHandler)

    logger = logging.getLogger("monitor")
    logger.setLevel(logging.INFO)

    level = "INFO"
    evento = "termino_run"
    value = "nome_notebook_executado"
    elapsed_time = 150
    error_msg = "Mensagem de erro, se houver"

    log_event(level, evt=evento, notebook=value, elapsed_time=elapsed_time, error=error_msg)
    
    """
    rec = logging.LogRecord("monitor", getattr(logging, level), "", 0, "", (), None)
    rec.payload = payload
    logger.handle(rec)


def determine_evt_level(error_msg, timeout_msg):
    """
    Função para determinar o evento e o nível com base nas mensagens de erro e timeout.
    """
    if error_msg and timeout_msg:
        return "erro_run_timeout", "ERROR"
    elif error_msg:
        return "erro_run", "ERROR"
    elif timeout_msg:
        return "warning_timeout", "WARNING"
    else:
        return "termino_run", "INFO"